In [ ]:
import csv
import gzip
import json
import math
import os
import sys
import random
import bisect
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import optuna

# -----------------------
# Configuration / Hardcoded Paths
# -----------------------
SAMPLES_CSV = "training_samples.csv"
OUT_DIR = "training_results"
BLACKLIST_BED = "tight_clusters_under_2500bp.bed" 
ARTEFACT_GENES_BED = "Artifact_Genes_Blacklist.bed"
N_TRIALS = 3000
SEED = 42

# -----------------------
# Constants
# -----------------------
GENOME_SIZE = 6.4e9
R_CORD = 2e-8
EPS = 1e-15
CORD_WEIGHT = 5.0

# Set seeds
random.seed(SEED)
np.random.seed(SEED)

# -----------------------
# Math Helpers
# -----------------------
def poisson_deviance(y_arr: np.ndarray, lam_arr: np.ndarray, weights: np.ndarray = None) -> float:
    y = np.asarray(y_arr, dtype=float)
    lam = np.asarray(lam_arr, dtype=float) + EPS
    
    with np.errstate(divide='ignore', invalid='ignore'):
        log_term = np.log(y / lam)
        term1 = np.where(y > 0, y * log_term, 0.0)
    
    dev = 2.0 * (term1 - (y - lam))
    
    if weights is not None:
        dev = dev * weights
        
    return float(np.sum(dev))

def expected_lambda(sample_type: str, age: float, bases: float) -> float:
    st = sample_type.lower()
    current_age = float(age) if age is not None else 0.0
    
    if "cord" in st:
        rate = R_CORD
    elif "pbmc" in st:
        rate = (16 * current_age + 105) / GENOME_SIZE
    else:
        rate = (22 * current_age + 164) / GENOME_SIZE
        
    return rate * bases

# -----------------------
# BED File Lookup Logic
# -----------------------
class BedLookup:
    def __init__(self, bed_path: str, name: str):
        self.intervals = {} 
        self.name = name
        self._load_bed(bed_path)

    def _load_bed(self, bed_path: str):
        if not os.path.exists(bed_path):
            print(f"[WARN] BED file not found: {bed_path}. {self.name} filter will do nothing.", file=sys.stderr)
            return

        print(f"[INFO] Loading {self.name} from {bed_path}...")
        count = 0
        with open(bed_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                chrom = parts[0]
                start, end = int(parts[1]), int(parts[2])
                
                if chrom not in self.intervals:
                    self.intervals[chrom] = []
                self.intervals[chrom].append((start, end))
                count += 1
        
        for chrom in self.intervals:
            self.intervals[chrom].sort(key=lambda x: x[0])
        print(f"[INFO] Loaded {count} regions for {self.name}.")

    def is_overlapped(self, chrom: str, pos: int) -> bool:
        if chrom not in self.intervals:
            return False
        
        regions = self.intervals[chrom]
        query_idx = pos - 1 
        
        idx = bisect.bisect_right(regions, (query_idx, float('inf')))
        
        for i in range(max(0, idx - 1), -1, -1):
            start, end = regions[i]
            if end <= query_idx:
                break 
            if start <= query_idx < end:
                return True
        return False

# -----------------------
# VCF Parsing to DataFrame
# -----------------------
def parse_vcf_to_df(vcf_path: str, sample_id: str) -> pd.DataFrame:
    data = []
    
    if not os.path.exists(vcf_path):
        print(f"[WARN] File not found: {vcf_path}", file=sys.stderr)
        return pd.DataFrame()

    openf = gzip.open if vcf_path.endswith(".gz") else open
    
    try:
        with openf(vcf_path, "rt") as fh:
            for line in fh:
                if line.startswith("#"):
                    continue
                fields = line.strip().split("\t")
                
                chrom = fields[0]
                pos = int(fields[1])
                
                info_str = fields[7]
                info_dict = {}
                for item in info_str.split(";"):
                    if "=" in item:
                        k, v = item.split("=", 1)
                        info_dict[k] = v
                    else:
                        info_dict[item] = "Yes" 
                
                info_dict["sample_id"] = sample_id
                info_dict["CHROM"] = chrom
                info_dict["POS"] = pos
                data.append(info_dict)
    except Exception as e:
        print(f"[ERROR] Error reading {vcf_path}: {e}", file=sys.stderr)
        return pd.DataFrame()

    if not data:
        return pd.DataFrame()

    df = pd.DataFrame(data)
    
    # --- Type Conversion ---
    def to_num(col, dtype, default):
        if col in df.columns:
            return pd.to_numeric(df[col], errors='coerce').fillna(default).astype(dtype)
        return pd.Series([default]*len(df), index=df.index, dtype=dtype)

    def to_str(col, default):
        if col in df.columns:
            return df[col].fillna(default)
        return pd.Series([default]*len(df), index=df.index)

    df['CHROM'] = to_str('CHROM', 'chr1')
    df['POS'] = to_num('POS', int, 0)

    # 1. Base Counts
    df['N_ALT_SSCS'] = to_num('N_ALT_SSCS', int, 0)
    df['N_TOTAL'] = to_num('N_TOTAL', int, 0)
    df['N_ALT'] = to_num('N_ALT', int, 0) 
    df['N_TOTAL_SSCS'] = to_num('N_TOTAL_SSCS', int, 0)
    df['N_TOTAL_SSCS'] = np.where(df['N_TOTAL_SSCS'] <= 0, df['N_TOTAL'], df['N_TOTAL_SSCS'])

    df['DIST_TO_INDEL'] = to_num('DIST_TO_INDEL', int, -1)
    df['HOMOPOLYMER_LEN'] = to_num('HOMOPOLYMER_LEN', int, 0)
    df['DEPTH_PERCENTILE'] = to_num('DEPTH_PERCENTILE', int, 50)

    # 2. Integer Metrics
    df['VARIANTS_20BP'] = to_num('VARIANTS_20BP', int, 0)
    df['VARIANTS_250BP'] = to_num('VARIANTS_250BP', int, 0)
    df['MEDIAN_DIST_TO_SOFTCLIP'] = to_num('MEDIAN_DIST_TO_SOFTCLIP', int, 1000)

    # 3. Float Metrics
    df['AVG_ASXS'] = to_num('AVG_ALT_ASXS', float, 0.0) 
    df['MEAN_MAPQ'] = to_num('MEAN_ALT_MAPQ', float, 0.0) 
    df['MEAN_BASEQ'] = to_num('MEAN_ALT_BASEQ', float, 0.0) 
    df['GC_CONTENT'] = to_num('GC_CONTENT', float, 0.5)
    df['STRAND_BIAS_SSCS'] = to_num('STRAND_BIAS_SSCS', float, 0.5)
    df['RP_SD_SSCS'] = to_num('RP_SD_SSCS', float, 0.0)
    df['RP_MED_SSCS'] = to_num('RP_MED_SSCS', float, 0.5)
    df['AVG_ALTREAD_VARIANTS'] = to_num('AVG_ALTREAD_VARIANTS', float, 0.0)
    df['REF_ENTROPY_40BP'] = to_num('REF_ENTROPY_40BP', float, 2.0)
    df['AVG_ALT_SOFTCLIP'] = to_num('AVG_ALT_SOFTCLIP', float, 0.0)
    df['AVG_ALT_N_COUNT'] = to_num('AVG_ALT_N_COUNT', float, 0.0)
    df['AVG_ALT_NM'] = to_num('AVG_ALT_NM', float, 0.0)
    df['MEAN_INSERT'] = to_num('MEAN_INSERT', float, 200.0)
    
    # --- ADDED: READ POSITION ---
    df['AVG_READ_POSITION'] = to_num('AVG_READ_POSITION', float, 75.0)

    # 4. Flags
    df['IN_CORD_PON'] = to_str('IN_CB_PON', "No")
    df['IN_GNOMAD_COMMON'] = to_str('IN_GNOMAD_COMMON', "No")
    df['IN_ENCODE_BLACKLIST'] = to_str('IN_ENCODE_BLACKLIST', "No")
    df['IN_REPEATMASKER'] = to_str('IN_REPEATMASKER', "No")

    # 5. VAFs
    with np.errstate(divide='ignore', invalid='ignore'):
        df['VAF_DCS'] = df['N_ALT'] / df['N_TOTAL']
        df['VAF_DCS'] = df['VAF_DCS'].fillna(0.0)
        df['VAF_SSCS'] = df['N_ALT_SSCS'] / df['N_TOTAL_SSCS']
        df['VAF_SSCS'] = df['VAF_SSCS'].fillna(0.0)

    return df

# -----------------------
# Vectorized Filtering
# -----------------------
def get_passing_mask(df: pd.DataFrame, thresholds: Dict[str, Any], pon_flags: Dict[str, bool], ignore_blacklists: bool = False) -> pd.Series:
    # 1. BASELINE HARD FILTERS
    mask = (df['N_ALT_SSCS'] > 1) & \
           (df['IN_ENCODE_BLACKLIST'] != "Yes") & \
           (df['IN_REPEATMASKER'] != "Yes")
    
    # --- ADDED: OxoG / Read Position Hard Filter ---
    # Removes variants too close to the start (<4) or end (>138) of the read
    mask &= (df['AVG_READ_POSITION'] >= 4) & (df['AVG_READ_POSITION'] <= 138)

    if pon_flags.get("use_cord_pon", False):
        mask &= (df['IN_CORD_PON'] != "Yes")

    if not ignore_blacklists:
        mask &= (df['IN_CLUSTER_BLACKLIST'] == False)
        mask &= (df['IN_ARTEFACT_BLACKLIST'] == False)

    # 2. DEPTH
    mask &= (df['N_TOTAL'] >= thresholds['n_total_min'])
    mask &= (df['DEPTH_PERCENTILE'] <= thresholds['depth_percentile_max'])

    # 3. QUALITY METRICS
    mask &= (df['AVG_ASXS'] > thresholds['asxs_min'])
    mask &= (df['MEAN_MAPQ'] >= thresholds['mean_mapq_min'])
    mask &= (df['MEAN_BASEQ'] >= thresholds['mean_baseq_min'])
    mask &= ((df['DIST_TO_INDEL'] == -1) | (df['DIST_TO_INDEL'] >= thresholds['dist_to_indel_min']))

    # 4. TUNABLE FILTERS
    mask &= (df['VARIANTS_20BP'] <= thresholds['variants_20bp_max'])
    mask &= (df['VARIANTS_250BP'] <= thresholds['variants_250bp_max'])
    mask &= (df['REF_ENTROPY_40BP'] >= thresholds['ref_entropy_min'])
    mask &= (df['AVG_ALT_SOFTCLIP'] <= thresholds['avg_alt_softclip_max'])
    mask &= (df['MEDIAN_DIST_TO_SOFTCLIP'] >= thresholds['dist_to_softclip_min'])
    mask &= (df['AVG_ALT_N_COUNT'] <= thresholds['avg_alt_n_count_max'])
    mask &= (df['AVG_ALT_NM'] <= thresholds['avg_alt_nm_max'])
    mask &= (df['MEAN_INSERT'] <= thresholds['mean_insert_max'])

    # 5. CONTEXT & STRAND
    mask &= (df['GC_CONTENT'] >= thresholds['gc_content_min']) & (df['GC_CONTENT'] <= thresholds['gc_content_max'])
    mask &= (df['HOMOPOLYMER_LEN'] <= thresholds['homopolymer_len_max'])

    # Strand Bias 
    enough_reads_sb = df['N_ALT_SSCS'] >= thresholds['strand_bias_n_alt_min']
    bad_bias = (df['STRAND_BIAS_SSCS'] < thresholds['strand_bias_min']) | (df['STRAND_BIAS_SSCS'] > thresholds['strand_bias_max'])
    mask &= ~(enough_reads_sb & bad_bias)

    # RP SD 
    enough_reads_rp = df['N_ALT_SSCS'] >= thresholds['rp_sd_min_alt']
    bad_rp_sd = df['RP_SD_SSCS'] < thresholds['rp_sd_min']
    mask &= ~(enough_reads_rp & bad_rp_sd)

    # RP MED
    mask &= (df['RP_MED_SSCS'] > thresholds['rp_med_min']) & (df['RP_MED_SSCS'] < thresholds['rp_med_max'])

    # 6. COMPLEX LOGIC
    cond1 = df['AVG_ALTREAD_VARIANTS'] > thresholds['avg_altread_variants_max']
    cond2 = (df['AVG_ALTREAD_VARIANTS'] > thresholds['avg_altread_variants_max_for_high_vaf']) & \
            (df['VAF_DCS'] > thresholds['avg_altread_high_vaf_cutoff'])
    mask &= ~(cond1 | cond2)

    # 7. GERMLINE & COMMON
    mask &= (df['VAF_SSCS'] <= thresholds['germline_vaf_max'])
    high_vaf_gnomad = (df['VAF_SSCS'] > thresholds['gnomad_vaf_cutoff'])
    is_common = (df['IN_GNOMAD_COMMON'] == "Yes") 
    mask &= ~(high_vaf_gnomad & is_common)

    return mask

# -----------------------
# Optimization Objective
# -----------------------
def make_objective(all_variants_df: pd.DataFrame, samples_meta: List[Dict]):
    
    expected_lambdas = []
    weights = []
    
    for s in samples_meta:
        expected_lambdas.append(expected_lambda(s['sample_type'], s['age'], s['bases']))
        
        if "cord" in s['sample_type'].lower():
            weights.append(CORD_WEIGHT)
        else:
            weights.append(1.0)
            
    exp_arr = np.array(expected_lambdas)
    weights_arr = np.array(weights)

    def objective(trial):
        thresholds = {
            "germline_vaf_max": trial.suggest_float("germline_vaf_max", 0.25, 0.35),
            "gnomad_vaf_cutoff": trial.suggest_float("gnomad_vaf_cutoff", 0.10, 0.25),
            "depth_percentile_max": trial.suggest_int("depth_percentile_max", 90, 99),
            "variants_20bp_max": trial.suggest_int("variants_20bp_max", 0, 4),
            "variants_250bp_max": trial.suggest_int("variants_250bp_max", 0, 15),
            "ref_entropy_min": trial.suggest_float("ref_entropy_min", 0.0, 2.1),
            "avg_alt_softclip_max": trial.suggest_float("avg_alt_softclip_max", 0.0, 2.0),
            "dist_to_softclip_min": trial.suggest_int("dist_to_softclip_min", 0, 5),
            "avg_alt_n_count_max": trial.suggest_float("avg_alt_n_count_max", 0.0, 3.0),
            "avg_alt_nm_max": trial.suggest_float("avg_alt_nm_max", 0.0, 5.0),
            "mean_insert_max": trial.suggest_float("mean_insert_max", 80, 800.0),
            "asxs_min": trial.suggest_int("asxs_min", 10, 60),
            "mean_mapq_min": trial.suggest_int("mean_mapq_min", 20, 60),
            "dist_to_indel_min": trial.suggest_int("dist_to_indel_min", 0, 500),
            "mean_baseq_min": trial.suggest_int("mean_baseq_min", 20, 120),
            "gc_content_min": trial.suggest_float("gc_content_min", 0.0, 0.3),
            "gc_content_max": trial.suggest_float("gc_content_max", 0.85, 1.0),
            "strand_bias_n_alt_min": trial.suggest_int("strand_bias_n_alt_min", 1, 20),
            "strand_bias_min": trial.suggest_float("strand_bias_min", 0.0, 0.3),
            "strand_bias_max": trial.suggest_float("strand_bias_max", 0.7, 1.0),
            "rp_sd_min": trial.suggest_float("rp_sd_min", 0.0, 0.1),
            "rp_sd_min_alt": trial.suggest_int("rp_sd_min_alt", 1, 10),
            "homopolymer_len_max": trial.suggest_int("homopolymer_len_max", 0, 12),
            "avg_altread_variants_max": trial.suggest_float("avg_altread_variants_max", 0, 10.0),
            "avg_altread_variants_max_for_high_vaf": trial.suggest_float("avg_altread_variants_max_for_high_vaf", 0, 10.0),
            "avg_altread_high_vaf_cutoff": trial.suggest_float("avg_altread_high_vaf_cutoff", 0.05, 0.35),
            "n_total_min": trial.suggest_int("n_total_min", 1, 30),
            "rp_med_min": trial.suggest_float("rp_med_min", 0.03, 0.1),
            "rp_med_max": trial.suggest_float("rp_med_max", 0.9, 0.97),
        }

        pon_flags = {"use_cord_pon": True}

        pass_mask = get_passing_mask(all_variants_df, thresholds, pon_flags)
        obs_series = all_variants_df[pass_mask].groupby('sample_id').size()
        obs_arr = np.array([obs_series.get(s['sample_id'], 0) for s in samples_meta])

        loss = poisson_deviance(obs_arr, exp_arr, weights=weights_arr)
        return loss

    return objective

# -----------------------
# Reporting
# -----------------------
def evaluate_and_report(best_thresholds, pon_flags, all_variants_df, samples_meta, outdir):
    os.makedirs(outdir, exist_ok=True)
    
    pass_mask_final = get_passing_mask(all_variants_df, best_thresholds, pon_flags, ignore_blacklists=False)
    pass_mask_clean = get_passing_mask(all_variants_df, best_thresholds, pon_flags, ignore_blacklists=True)
    
    variants_saved_by_blacklist = pass_mask_clean & (~pass_mask_final)
    count_saved_by_blacklist = variants_saved_by_blacklist.sum()

    obs_series = all_variants_df[pass_mask_final].groupby('sample_id').size()
    
    observed = []
    expected = []
    ids = []
    ages = []
    bases = []
    types = []
    weights = []

    for s in samples_meta:
        sid = s['sample_id']
        obs = obs_series.get(sid, 0)
        lam = expected_lambda(s['sample_type'], s['age'], s['bases'])
        
        ids.append(sid)
        observed.append(obs)
        expected.append(lam)
        ages.append(s['age'])
        bases.append(s['bases'])
        types.append(s['sample_type'])
        
        if "cord" in s['sample_type'].lower():
            weights.append(CORD_WEIGHT)
        else:
            weights.append(1.0)

    obs_arr = np.array(observed)
    exp_arr = np.array(expected)
    weights_arr = np.array(weights)

    with open(os.path.join(outdir, "best_thresholds.json"), "w") as fh:
        config_out = {
            "thresholds": best_thresholds, 
            "pon_flags": pon_flags,
            "blacklists_enforced": True,
            "cord_penalty_weight": CORD_WEIGHT
        }
        json.dump(config_out, fh, indent=2)

    ratio = obs_arr / (exp_arr + EPS)
    dev = poisson_deviance(obs_arr, exp_arr, weights=weights_arr)
    mean_ratio = float(np.mean(ratio))
    
    summary = {
        "n_samples": len(samples_meta),
        "obs_mean_over_expected": mean_ratio,
        "total_observed": int(np.sum(obs_arr)),
        "total_expected": float(np.sum(exp_arr)),
        "weighted_poisson_deviance": dev,
        "variants_blocked_by_blacklists": int(count_saved_by_blacklist)
    }
    with open(os.path.join(outdir, "summary.json"), "w") as fh:
        json.dump(summary, fh, indent=2)

    res_df = pd.DataFrame({
        "sample_id": ids,
        "sample_type": types,
        "age": ages,
        "bases": bases,
        "observed": observed,
        "expected": expected,
        "weight_used": weights 
    })
    res_df.to_csv(os.path.join(outdir, "observed_expected.csv"), index=False)
    
    print("-" * 30)
    print("FINAL RESULTS")
    print("-" * 30)
    print(f"Total Samples:      {len(samples_meta)}")
    print(f"Total Observed SNVs: {int(np.sum(obs_arr))}")
    print(f"Total Expected SNVs: {float(np.sum(exp_arr)):.2f}")
    print(f"Weighted Deviance:  {dev:.4f} (Cord weight = {CORD_WEIGHT}x)")
    print(f"Mean Obs/Exp Ratio: {mean_ratio:.4f}")
    print(f"Blacklist Blocked:  {count_saved_by_blacklist} variants (passed filters but hit blacklist)")
    print(f"Results saved to:   {outdir}")
    print("-" * 30)

# -----------------------
# Execution Block
# -----------------------
if __name__ == "__main__":
    print(f"[INFO] Reading metadata from {SAMPLES_CSV}...")
    samples_meta = []
    with open(SAMPLES_CSV, newline='') as csvfile:
        rdr = csv.DictReader(csvfile)
        for r in rdr:
            samples_meta.append({
                "sample_id": r["sample_id"],
                "vcf_path": r["vcf_path"],
                "sample_type": r["sample_type"].lower(),
                "age": float(r["age"]) if r.get("age") else None,
                "bases": float(r["bases"]),
                "mean_dcs_depth": float(r.get("mean_dcs_depth", 0)),
            })
    print(f"[INFO] Loaded {len(samples_meta)} samples.")

    print("[INFO] Loading VCFs into DataFrame...")
    df_list = []
    for s in samples_meta:
        df_list.append(parse_vcf_to_df(s["vcf_path"], s["sample_id"]))

    if df_list:
        all_variants_df = pd.concat(df_list, ignore_index=True)
    else:
        print("[ERROR] No VCF data loaded. Checking paths in CSV.")
        all_variants_df = pd.DataFrame()

    print(f"[INFO] Total variants loaded: {len(all_variants_df)}")

    if not all_variants_df.empty:
        bed_lookup_cluster = BedLookup(BLACKLIST_BED, "Cluster Blacklist")
        all_variants_df['IN_CLUSTER_BLACKLIST'] = all_variants_df.apply(
            lambda row: bed_lookup_cluster.is_overlapped(row['CHROM'], row['POS']), axis=1
        )
        
        bed_lookup_genes = BedLookup(ARTEFACT_GENES_BED, "Artefact Gene Blacklist")
        all_variants_df['IN_ARTEFACT_BLACKLIST'] = all_variants_df.apply(
            lambda row: bed_lookup_genes.is_overlapped(row['CHROM'], row['POS']), axis=1
        )
        
        hits_cluster = all_variants_df['IN_CLUSTER_BLACKLIST'].sum()
        hits_genes = all_variants_df['IN_ARTEFACT_BLACKLIST'].sum()
        print(f"[INFO] Found {hits_cluster} variants in Cluster blacklist.")
        print(f"[INFO] Found {hits_genes} variants in Artefact Gene blacklist.")

        print(f"[INFO] Starting Optuna (Trials={N_TRIALS})...")
        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
        
        objective = make_objective(all_variants_df, samples_meta)
        study.optimize(objective, n_trials=N_TRIALS)

        print(f"[INFO] Best Weighted Deviance: {study.best_value:.4f}")

        best_params = study.best_params
        best_thresholds = {k: v for k, v in best_params.items()}
        pon_flags = {"use_cord_pon": True}
        
        print("[INFO] Generating report...")
        evaluate_and_report(best_thresholds, pon_flags, all_variants_df, samples_meta, OUT_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# -----------------------
# Configuration
# -----------------------
# Path to the CSV generated in the previous step
CSV_PATH = "training_results/observed_expected.csv"
OUTPUT_PNG = "training_results/observed_vs_expected_plot.png"

# Check if file exists
if not os.path.exists(CSV_PATH):
    print(f"[ERROR] Could not find {CSV_PATH}. Please run the optimization cell first.")
else:
    # Load table
    df = pd.read_csv(CSV_PATH)

    # Convert data arrays
    ages = df["age"].values
    bases = df["bases"].values
    observed = df["observed"].values
    expected = df["expected"].values
    sample_types = df["sample_type"].values

    # Calculate SNV rates per base
    obs_rate = observed / bases
    exp_rate = expected / bases

    # -----------------------
    # Plotting
    # -----------------------
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot Observed (Blue circles)
    ax.scatter(ages, obs_rate, label="Observed", color="#1f77b4", alpha=0.8, marker="o", s=60, edgecolors='k', linewidth=0.5)

    # Plot Expected (Orange crosses)
    ax.scatter(ages, exp_rate, label="Expected", color="#ff7f0e", alpha=0.8, marker="x", s=60, linewidth=2)

    # Styling
    ax.set_xlabel("Age (years)", fontsize=12)
    ax.set_ylabel("SNV rate (SNVs per base)", fontsize=12)
    
    # Handle X-limits (pad slightly around min/max age)
    # If ages has None/NaN (Cord blood sometimes represented as negative or 0), handle gracefully
    valid_ages = ages[~np.isnan(ages)]
    if len(valid_ages) > 0:
        ax.set_xlim(min(valid_ages) - 2, max(valid_ages) + 5)
    else:
        ax.set_xlim(-2, 100)

    # Hardcoded Y-limits from your request
    ax.set_ylim(0, 5.5e-7)
    
    ax.set_title("Observed vs Expected SNV Rate (Training Set)", fontsize=14)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)
    
    # Legend
    ax.legend(loc='upper left', frameon=True, fontsize=10)

    plt.tight_layout()
    
    # Save and Show
    plt.savefig(OUTPUT_PNG, dpi=150)
    print(f"[INFO] Plot saved to: {OUTPUT_PNG}")
    plt.show()